### 利用深度信息去除背景

In [ ]:
import zivid
import open3d as o3d
import numpy as np
from pathlib import Path

#读取原始的zdf文件
zdf_path = Path(r"D:\captures_stop_and_go\forward\capture_003.zdf")

with zivid.Application() as app:   #初始化sdk环境
    frame = zivid.Frame(str(zdf_path)) #读取zdf文件，frame可以理解为一帧3d采集的结果
    point_cloud = frame.point_cloud() #从这一帧中获取点云对象
    print("zdf文件读取成功")

    #读取xyz坐标，单位为mm
    xyz = point_cloud.copy_data("xyz")
    #读取颜色
    rgba = point_cloud.copy_data("rgba")
    print("坐标和颜色读取成功")

    # 显示原始点云
    valid = np.isfinite(xyz).all(axis=2)
    pcd_original = o3d.geometry.PointCloud()
    pcd_original.points = o3d.utility.Vector3dVector(xyz[valid].astype(np.float64) / 1000.0)
    pcd_original.colors = o3d.utility.Vector3dVector(rgba[valid, :3].astype(np.float64) / 255.0)
    o3d.visualization.draw_geometries([pcd_original])

#将二维图像形式的点云转换成N*3格式，只是改变numpy数组的格式
points = xyz.reshape(-1, 3)
colors = rgba[:, :, :3].reshape(-1, 3)  #[:, :, :3]保留所有行和列，但只保留RGB三个通道并舍弃alpha通道
print("点云转换成功")

#根据z深度过滤，单位mm
z_min = 1100
z_max = 1500
mask = ( np.isfinite(points).all(axis=1) & (points[:, 2] >= z_min) & (points[:, 2] <= z_max)) #排除无效的坐标并只提取所有点的第三列（也就是z轴）的值进行比较

filtered_points = points[mask]
filtered_colors = colors[mask]
print("过滤成功")

#转换为open3D点云
pcd = o3d.geometry.PointCloud()

#mm转换成m
pcd.points = o3d.utility.Vector3dVector( filtered_points.astype(np.float64) / 1000.0)
pcd.colors = o3d.utility.Vector3dVector( filtered_colors.astype(np.float64) / 255.0)

#显示结果
o3d.visualization.draw_geometries([pcd])


#获取原始zdf所在文件夹
output_folder = zdf_path.parent
output_path = output_folder / f"{zdf_path.stem}_filtered.ply"  #保存的文件名为capture_001_filtered.ply
#保存过滤后的点云
o3d.io.write_point_cloud(str(output_path), pcd)
print(f"过滤后的点云已保存到: {output_path}")

zdf文件读取成功
坐标和颜色读取成功
点云转换成功
过滤成功
过滤后的点云已保存到: D:\captures_stop_and_go\forward\capture_003_filtered.ply
